# Исследования к решению

Всё, что обосновывает решение: аудит данных, baseline-ы, вклад групп признаков, выбор модели, абляция, проверка сдвига,
отклонённые идеи и разбор ошибок. Первая ячейка запускает `solution.ipynb` и берёт из него данные, признаки и функции
валидации (финальная модель переобучается, `submission.csv` пересоздаётся тем же). Выводы — в `README.md`.

1. Аудит данных
2. Baseline-ы
3. Признаки: пропуски, сигнал history, сила отдельных признаков
4. Вклад групп признаков
5. Гиперпараметры
6. Выбор модели: ансамбль против CatBoost
7. Абляция и важность признаков
8. Сдвиг между train и test
9. Отклонённые идеи
10. Разбор ошибок
11. Сводная таблица

In [1]:
# данные, признаки, валидация и финальная модель — из решения
%run solution.ipynb

Python 3.14.4 | pandas 3.0.6 | scikit-learn 1.9.1 | catboost 1.2.10


train (11091, 5) | test (4909, 4) | events (328905, 14)


событий: было 328905 → стало 283830


признаков: 74 {'timing': 21, 'content': 28, 'pointer': 9, 'client': 5, 'cookie': 2, 'history': 4, 'reputation': 5}


submission.csv: 4909 строк, уникальных score: 4909


In [2]:
from scipy.stats import rankdata
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from metric import pr_curve

pd.set_option('display.max_colwidth', 120)


def _bootstrap(y_true, s_a, s_b, rng, n):
    diffs = []
    for _ in range(n):
        i = rng.integers(0, len(y_true), len(y_true))
        diffs.append(precision_at_recall(y_true[i], s_a[i]) - precision_at_recall(y_true[i], s_b[i]))
    diffs = np.array(diffs)
    return {'разница': diffs.mean(), 'от': np.percentile(diffs, 2.5), 'до': np.percentile(diffs, 97.5),
            'P(разница > 0)': (diffs > 0).mean()}


def bootstrap_diff(a, b, n=1000):
    """Парный бутстрап по кукам: разница P@R0.7 экспериментов a и b (95% интервал) на OOF и на каждом блоке."""
    rng = np.random.default_rng(SEED)
    m = oof[a].notna().to_numpy()
    out = {'OOF 8 дней': _bootstrap(y[m], oof[a][m].to_numpy(), oof[b][m].to_numpy(), rng, n)}
    for start in BLOCK_STARTS:
        _, va = block_masks(start)
        out[f'неделя с {start:%d.%m}'] = _bootstrap(y[va], block_scores[a][start], block_scores[b][start], rng, n)
    return pd.DataFrame(out).T.round(3)

## 1. Аудит данных

In [3]:
# уникальность кук, пересечение train/test, длина окна, период: тест лежит строго позже трейна
print('дубли cookie_id в train / test:', train.cookie_id.duplicated().sum(), '/', test.cookie_id.duplicated().sum())
print('пересечение train и test:', len(set(train.cookie_id) & set(test.cookie_id)))
print('длина окна:', (meta.window_end_ts - meta.window_start_ts).value_counts().to_dict())
print('train:', train.window_start_ts.min().date(), '—', train.window_start_ts.max().date())
print('test: ', test.window_start_ts.min().date(), '—', test.window_start_ts.max().date())
print('доля ботов в train:', round(train.target.mean(), 4))

train.groupby(train.window_start_ts.dt.date).target.agg(n_cookies='size', bot_rate='mean').round(3).T

дубли cookie_id в train / test: 0 / 0
пересечение train и test: 0
длина окна: {Timedelta('1 days 00:00:00'): 16000}
train: 2026-04-06 — 2026-04-19
test:  2026-04-20 — 2026-04-26
доля ботов в train: 0.0811


window_start_ts,2026-04-06,2026-04-07,2026-04-08,2026-04-09,2026-04-10,2026-04-11,2026-04-12,2026-04-13,2026-04-14,2026-04-15,2026-04-16,2026-04-17,2026-04-18,2026-04-19
n_cookies,772.000,797.000,834.000,902.000,912.00,896.000,817.000,843.000,826.000,801.000,740.000,713.000,606.000,632.000
bot_rate,0.088,0.074,0.084,0.073,0.09,0.088,0.082,0.063,0.085,0.081,0.081,0.093,0.086,0.066


In [4]:
# порядок строк, дубликаты, события вне окна, варианты platform, соответствие eid ↔ event_name, пропуски
print('отсортированы по времени:', events_raw.event_ts.is_monotonic_increasing)
print('полных дубликатов строк:', events_raw.duplicated().sum())

ev_meta = events_raw[['cookie_id', 'event_ts']].merge(
    meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id', how='left')
print('событий без куки в train/test:', ev_meta.window_start_ts.isna().sum())
print('событий до начала окна:', (ev_meta.event_ts < ev_meta.window_start_ts).sum())
print('событий после конца окна:', (ev_meta.event_ts >= ev_meta.window_end_ts).sum(),
      f'({(ev_meta.event_ts >= ev_meta.window_end_ts).mean():.1%})')

print('\nplatform:', events_raw.platform.value_counts().to_dict())
print('\neid ↔ event_name:', events_raw.groupby('eid').event_name.unique().map(list).to_dict())

events_raw.isna().mean().round(3).rename('доля пропусков').to_frame().T

отсортированы по времени: False


полных дубликатов строк: 4863
событий без куки в train/test: 0
событий до начала окна: 0
событий после конца окна: 40779 (12.4%)

platform: {'desktop': 46866, 'WEB': 46476, 'Web': 46365, 'web': 45951, 'ANDROID': 42462, 'Android': 42254, 'android': 42159, 'iphone': 4103, 'IOS': 4100, 'iOS': 4091, 'ios': 4078}

eid ↔ event_name: {100: ['search_results_view'], 200: ['item_view'], 210: ['photo_swipe'], 220: ['seller_page_view'], 300: ['contact_phone_show'], 301: ['contact_chat_open'], 303: ['contact_message_sent'], 400: ['favorite_add'], 500: ['login'], 900: ['captcha_shown']}


,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
доля пропусков,0.0,0.0,0.0,0.0,0.0,0.0,0.348,0.1,0.072,0.407,0.695,0.695,0.67,0.67


In [5]:
# регистр platform меняется внутри одной куки — шум логирования; после очистки у каждой куки одна платформа
raw_platform_per_cookie = events_raw.groupby('cookie_id').platform.nunique()
print('разных написаний platform у одной куки:', raw_platform_per_cookie.value_counts().sort_index().to_dict())
print('событий: было', len(events_raw), '→ стало после очистки', len(events))
print('платформ на куку после нормализации:', events.groupby('cookie_id').platform.nunique().value_counts().to_dict())

разных написаний platform у одной куки: {1: 444, 2: 1312, 3: 7144, 4: 7100}
событий: было 328905 → стало после очистки 283830
платформ на куку после нормализации: {1: 16000}


In [6]:
# разобранный User-Agent: тип клиента и семейство браузера для ~150 уникальных строк
user_agent_table(events_raw.user_agent.unique()).groupby(['ua_client', 'ua_family']).size().rename('n_ua_strings').to_frame()

n_ua_strings
ua_client       ua_family                    
app             avito_app                  33
desktop_browser chrome                     28
                firefox                    14
                yandex                     12
headless        headless_chrome             4
mobile_browser  chrome                     48
                safari                      3
script          curl                        1
                go-http-client              1
                node-fetch                  1
                python-requests             1
                python-urllib3              1
                scrapy                      1

## 2. Baseline-ы

In [7]:
# константа и RF из quickstart уже посчитаны в решении; здесь — одиночные эвристики и сумма рангов трёх эвристик
WINDOW_SECONDS = 24 * 3600   # одно событие — интервала нет: считаем такую куку максимально медленной


def heuristic(column, sign=1, fill=0.0):
    """Скор = значение одного признака (sign=-1 — чем меньше, тем подозрительнее)."""
    return lambda X_tr, y_tr, X_va: sign * X_va[column].fillna(fill).to_numpy()


def rank_rule(X_tr, y_tr, X_va):
    rank = lambda s: s.rank(pct=True)
    return (rank(X_va.n_events) + rank(-X_va.dt_median.fillna(WINDOW_SECONDS))
            + rank(-X_va.engagement_share)).to_numpy()


evaluate('эвристика: n_events', heuristic('n_events'), X)
evaluate('эвристика: -dt_median', heuristic('dt_median', sign=-1, fill=WINDOW_SECONDS), X)
evaluate('эвристика: -engagement_share', heuristic('engagement_share', sign=-1), X)
evaluate('правило: сумма рангов', rank_rule, X)
show_results()

,P@R0.7,P@R сглаж.,P@R0.7 по дням,PR-AUC,неделя с 11.04,неделя с 13.04
константа,0.079,0.079,0.080,0.079,0.082,0.079
baseline из quickstart (RF на 2 признаках),0.093,0.095,0.097,0.206,0.096,0.093
"финальная модель: CatBoost, 3 seed",0.913,0.877,0.909,0.836,0.892,0.894
эвристика: n_events,0.117,0.114,0.116,0.174,0.113,0.109
эвристика: -dt_median,0.184,0.177,0.173,0.299,0.154,0.179
эвристика: -engagement_share,0.131,0.125,0.134,0.122,0.132,0.136
правило: сумма рангов,0.140,0.149,0.152,0.442,0.145,0.139


## 3. Признаки: пропуски, сигнал history, сила отдельных признаков

In [8]:
# пропуски структурные: курсор только на web, глубина выдачи только у поиска и т. п.
print('признаков без меток:', X_all.shape[1], {k: len(v) for k, v in FEATURE_GROUPS.items() if k != 'reputation'})
X_all.isna().mean().loc[lambda s: s > 0].round(3).sort_values().rename('доля NaN').to_frame().T

признаков без меток: 69 {'timing': 21, 'content': 28, 'pointer': 9, 'client': 5, 'cookie': 2, 'history': 4}


,item_repeat_share,ua_major,pro_seller_share,dt_min,dt_top_value_share,dt_share_le5,dt_share_ge300,dt_p90,dt_p10,dt_mean,...,items_per_search,pointer_share,pointer_x_mean,pointer_round100_share,pointer_unique_share,pointer_y_mean,pointer_edge_share,pointer_x_std,pointer_y_std,pointer_step_median
доля NaN,0.015,0.017,0.021,0.022,0.022,0.022,0.022,0.022,0.022,0.022,...,0.085,0.464,0.634,0.634,0.634,0.634,0.634,0.644,0.644,0.644


In [9]:
# сигнал history: боты разных кук смотрят одни и те же объявления — доля пар «бот–бот» среди пар кук с общим
# объявлением в один день против ожидаемой при случайных интересах; ниже — медианы history на днях с полной историей
labeled_views = values_by_day(events, meta).assign(t=lambda d: d.cookie_id.map(train_labels)).dropna(subset=['t'])
pairs = labeled_views.merge(labeled_views, on=['day', 'item_id']).query('cookie_id_x != cookie_id_y')
bot_bot_share = ((pairs.t_x == 1) & (pairs.t_y == 1)).mean()
print(f'пары кук с общим объявлением в один день: бот–бот {bot_bot_share:.1%}, '
      f'при случайных интересах ожидалось бы ≈{train.target.mean() ** 2:.1%}')

full_history = (day >= VALID_DAYS[0] + pd.Timedelta(days=1)).to_numpy()   # дни с полной недельной историей
X[full_history].groupby(y[full_history])[FEATURE_GROUPS['history']].median().rename(index={0: 'люди', 1: 'боты'}).round(3)

пары кук с общим объявлением в один день: бот–бот 10.2%, при случайных интересах ожидалось бы ≈0.7%


,hist_pop_mean,hist_pop_median,hist_pop_max,hist_seen_share
люди,0.167,0.167,0.374,0.600
боты,0.319,0.332,0.686,0.833


In [10]:
# разделяющая сила каждого признака отдельно (дни с 12 апреля, где у признаков на истории уже есть прошлое);
# ни один не близок к 1 — признака-утечки нет
late = (day >= VALID_DAYS[0]).to_numpy()
single_auc = pd.Series({c: roc_auc_score(y[late], X_full[c][late].fillna(X_full[c].median()))
                        for c in X_full.select_dtypes('number').columns})
single_auc = (single_auc - 0.5).abs().add(0.5).sort_values(ascending=False)
single_auc.head(15).round(3).rename('ROC-AUC').to_frame().T

,item_bot_seen_count,item_bot_seen_share,item_rep_mean,item_rep_max,hist_pop_mean,item_rep_evidence,hist_pop_max,hist_pop_median,dt_median,location_entropy,location_nunique,hist_seen_share,session_events_max,events_per_active_min,searches_per_query
ROC-AUC,0.798,0.797,0.794,0.783,0.766,0.76,0.755,0.746,0.743,0.726,0.719,0.708,0.707,0.702,0.692


## 4. Вклад групп признаков

In [11]:
# логистическая регрессия — линейная точка отсчёта; HistGradientBoosting — быстрый бустинг,
# на котором сравниваются наборы признаков: поведение → + history → + reputation
def numeric_columns(X):
    return [c for c in X.columns if c not in CATEGORICAL]


def logreg(X_tr, y_tr, X_va):
    signed_log = FunctionTransformer(lambda a: np.sign(a) * np.log1p(np.abs(a)))   # сжимаем тяжёлые хвосты
    prep = ColumnTransformer([
        ('num', make_pipeline(signed_log, SimpleImputer(strategy='median', add_indicator=True), StandardScaler()),
         numeric_columns(X_tr)),
        ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL),
    ])
    model = make_pipeline(prep, LogisticRegression(C=0.1, max_iter=2000))
    return model.fit(X_tr, y_tr).predict_proba(X_va)[:, 1]


def make_hgb(**params):
    base = dict(learning_rate=0.05, max_iter=300, max_leaf_nodes=15, min_samples_leaf=40, l2_regularization=1.0,
                categorical_features='from_dtype', random_state=SEED)
    return HistGradientBoostingClassifier(**{**base, **params})


def hgb(**params):
    """fit_predict для бустинга с заданными гиперпараметрами."""
    return lambda X_tr, y_tr, X_va: make_hgb(**params).fit(X_tr, y_tr).predict_proba(X_va)[:, 1]


BEHAVIOR = [c for c in X.columns if c not in FEATURE_GROUPS['history']]
FINAL_HGB = 'бустинг: поведение + history + reputation'
evaluate('логистическая регрессия', logreg, X_full, X_full_blocks)
evaluate('бустинг: поведение', hgb(), X[BEHAVIOR])
evaluate('бустинг: поведение + history', hgb(), X)
evaluate(FINAL_HGB, hgb(), X_full, X_full_blocks)
show_results()

,P@R0.7,P@R сглаж.,P@R0.7 по дням,PR-AUC,неделя с 11.04,неделя с 13.04
константа,0.079,0.079,0.080,0.079,0.082,0.079
baseline из quickstart (RF на 2 признаках),0.093,0.095,0.097,0.206,0.096,0.093
"финальная модель: CatBoost, 3 seed",0.913,0.877,0.909,0.836,0.892,0.894
эвристика: n_events,0.117,0.114,0.116,0.174,0.113,0.109
эвристика: -dt_median,0.184,0.177,0.173,0.299,0.154,0.179
эвристика: -engagement_share,0.131,0.125,0.134,0.122,0.132,0.136
правило: сумма рангов,0.140,0.149,0.152,0.442,0.145,0.139
логистическая регрессия,0.871,0.832,0.869,0.794,0.839,0.867
бустинг: поведение,0.751,0.738,0.754,0.786,0.716,0.737
бустинг: поведение + history,0.849,0.825,0.846,0.814,0.771,0.788


In [12]:
# парный бутстрап: вклад history и reputation
pd.concat({'+ history к поведению': bootstrap_diff('бустинг: поведение + history', 'бустинг: поведение'),
           '+ reputation к history': bootstrap_diff(FINAL_HGB, 'бустинг: поведение + history')})

разница     от     до  P(разница > 0)
+ history к поведению  OOF 8 дней        0.099  0.050  0.151           1.000
                       неделя с 11.04    0.065  0.013  0.131           0.992
                       неделя с 13.04    0.060 -0.004  0.124           0.970
+ reputation к history OOF 8 дней        0.050  0.012  0.101           0.997
                       неделя с 11.04    0.074  0.027  0.122           0.999
                       неделя с 13.04    0.105  0.051  0.169           1.000

In [13]:
# цена устаревания меток (на тесте им 1–7 дней): неделя 13–19, репутация по меткам до начала недели
# против «свежих» меток до вчерашнего дня
start = BLOCK_STARTS[1]
tr, va = block_masks(start)
stale = hgb()(X_full_blocks[start][tr], y[tr], X_full_blocks[start][va])
fresh = hgb()(X_full[tr], y[tr], X_full[va])     # у обучающих строк метки те же; отличаются только строки недели
pd.DataFrame({'метки до начала недели (как на тесте)': {'P@R0.7': precision_at_recall(y[va], stale), 'P@R сглаж.': smoothed_par(y[va], stale)},
              'свежие метки до вчерашнего дня': {'P@R0.7': precision_at_recall(y[va], fresh), 'P@R сглаж.': smoothed_par(y[va], fresh)}}).T.round(3)

,P@R0.7,P@R сглаж.
метки до начала недели (как на тесте),0.892,0.857
свежие метки до вчерашнего дня,0.908,0.859


## 5. Гиперпараметры

In [14]:
# в v1 параметры подбирались на 3 фолдах по 3 дня; та же конфигурация — по старой и по новым схемам валидации
OLD_FOLDS = [('2026-04-11', '2026-04-13'), ('2026-04-14', '2026-04-16'), ('2026-04-17', '2026-04-19')]


def old_scheme_score(fit_predict, X):
    """Средний P@R0.7 на 3 фолдах по 3 дня — схема валидации из первой версии решения."""
    scores = []
    for fold_start, fold_end in OLD_FOLDS:
        tr, va = (day < fold_start).to_numpy(), day.between(fold_start, fold_end).to_numpy()
        scores.append(precision_at_recall(y[va], fit_predict(X[tr], y[tr], X[va])))
    return np.mean(scores)


V1_PARAMS = dict(learning_rate=0.06, max_leaf_nodes=31, min_samples_leaf=20, max_iter=400)
V1 = 'v1: бустинг с подобранными параметрами, только поведение'
evaluate(V1, hgb(**V1_PARAMS), X[BEHAVIOR])
pd.DataFrame({
    'по умолчанию': {'3 фолда × 3 дня': old_scheme_score(hgb(), X[BEHAVIOR]), **results['бустинг: поведение']},
    'подобранные на 3 фолдах (v1)': {'3 фолда × 3 дня': old_scheme_score(hgb(**V1_PARAMS), X[BEHAVIOR]), **results[V1]},
}).T.astype(float).round(3)

,3 фолда × 3 дня,P@R0.7,P@R сглаж.,P@R0.7 по дням,PR-AUC,неделя с 11.04,неделя с 13.04
по умолчанию,0.720,0.751,0.738,0.754,0.786,0.716,0.737
подобранные на 3 фолдах (v1),0.779,0.760,0.733,0.755,0.779,0.749,0.728


In [15]:
# две разумные альтернативы параметрам по умолчанию (участники ансамбля)
HGB_CONFIGS = {
    'бустинг: мелкие деревья': dict(max_leaf_nodes=7, max_iter=500),
    'бустинг: сильнее L2': dict(l2_regularization=5.0),
}
for name, params in HGB_CONFIGS.items():
    evaluate(name, hgb(**params), X_full, X_full_blocks)
show_results([FINAL_HGB, *HGB_CONFIGS])

,P@R0.7,P@R сглаж.,P@R0.7 по дням,PR-AUC,неделя с 11.04,неделя с 13.04
бустинг: поведение + history + reputation,0.902,0.862,0.912,0.828,0.859,0.892
бустинг: мелкие деревья,0.888,0.862,0.897,0.831,0.863,0.867
бустинг: сильнее L2,0.902,0.864,0.906,0.832,0.863,0.883


## 6. Выбор модели: ансамбль против CatBoost

In [16]:
# ансамбль: три бустинга и ExtraTrees (ошибки меньше коррелируют с бустингом); скоры разного масштаба,
# поэтому усредняются ранги — метрике нужен только порядок кук
def extra_trees(X_tr, y_tr, X_va):
    # деревьям не нужны масштабирование и логарифм; пропуск кодируем -1 — все числовые признаки неотрицательны
    prep = ColumnTransformer([('num', SimpleImputer(strategy='constant', fill_value=-1), numeric_columns(X_tr)),
                              ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL)])
    model = make_pipeline(prep, ExtraTreesClassifier(n_estimators=500, min_samples_leaf=3, max_features=0.3,
                                                     n_jobs=-1, random_state=SEED))
    model.fit(X_tr, y_tr)
    # параллельный predict складывает вероятности деревьев в случайном порядке (разница ~1e-16),
    # что меняет разбиение ничьих в ансамбле; в один поток результат воспроизводим бит в бит
    model[-1].set_params(n_jobs=1)
    return model.predict_proba(X_va)[:, 1]


ENSEMBLE_MEMBERS = {
    'бустинг': hgb(),
    **{name: hgb(**params) for name, params in HGB_CONFIGS.items()},
    'ExtraTrees': extra_trees,
}


def ensemble(X_tr, y_tr, X_va):
    scores = np.array([fit_predict(X_tr, y_tr, X_va) for fit_predict in ENSEMBLE_MEMBERS.values()])
    mean_rank = np.mean([rankdata(s) for s in scores], axis=0)
    # суммы целых рангов часто совпадают, а метрика не разделяет равные score;
    # ничьи разбиваем средним скором моделей, остальной порядок задаёт средний ранг
    order = np.lexsort((scores.mean(axis=0), mean_rank))
    final_rank = np.empty(len(order))
    final_rank[order] = np.arange(1, len(order) + 1)
    return final_rank / len(order)


evaluate('ExtraTrees', extra_trees, X_full, X_full_blocks)
evaluate('ансамбль', ensemble, X_full, X_full_blocks)
show_results([FINAL_HGB, *HGB_CONFIGS, 'ExtraTrees', 'ансамбль'])

,P@R0.7,P@R сглаж.,P@R0.7 по дням,PR-AUC,неделя с 11.04,неделя с 13.04
бустинг: поведение + history + reputation,0.902,0.862,0.912,0.828,0.859,0.892
бустинг: мелкие деревья,0.888,0.862,0.897,0.831,0.863,0.867
бустинг: сильнее L2,0.902,0.864,0.906,0.832,0.863,0.883
ExtraTrees,0.899,0.856,0.885,0.809,0.887,0.893
ансамбль,0.889,0.854,0.922,0.831,0.885,0.897


In [17]:
# CatBoost: финальная модель посчитана в решении (FINAL); для контроля — CatBoost полностью по умолчанию
def catboost_default(X_tr, y_tr, X_va):
    model = catboost_model(verbose=0, allow_writing_files=False)
    return model.fit(as_catboost_input(X_tr), y_tr).predict_proba(as_catboost_input(X_va))[:, 1]


evaluate('CatBoost по умолчанию', catboost_default, X_full, X_full_blocks)
show_results([FINAL_HGB, 'ансамбль', 'CatBoost по умолчанию', FINAL])

,P@R0.7,P@R сглаж.,P@R0.7 по дням,PR-AUC,неделя с 11.04,неделя с 13.04
бустинг: поведение + history + reputation,0.902,0.862,0.912,0.828,0.859,0.892
ансамбль,0.889,0.854,0.922,0.831,0.885,0.897
CatBoost по умолчанию,0.910,0.870,0.904,0.835,0.881,0.892
"финальная модель: CatBoost, 3 seed",0.913,0.877,0.909,0.836,0.892,0.894


In [18]:
# парный бутстрап финальной модели против альтернатив
pd.concat({'CatBoost против ансамбля': bootstrap_diff(FINAL, 'ансамбль'),
           'CatBoost против бустинга': bootstrap_diff(FINAL, FINAL_HGB),
           'CatBoost против v1': bootstrap_diff(FINAL, V1)})

разница     от     до  P(разница > 0)
CatBoost против ансамбля OOF 8 дней        0.018 -0.008  0.048           0.904
                         неделя с 11.04    0.009 -0.020  0.044           0.686
                         неделя с 13.04   -0.001 -0.027  0.023           0.471
CatBoost против бустинга OOF 8 дней        0.006 -0.020  0.028           0.696
                         неделя с 11.04    0.037 -0.002  0.085           0.971
                         неделя с 13.04    0.002 -0.032  0.041           0.504
CatBoost против v1       OOF 8 дней        0.147  0.083  0.208           1.000
                         неделя с 11.04    0.142  0.073  0.230           1.000
                         неделя с 13.04    0.164  0.091  0.242           1.000

## 7. Абляция и важность признаков

In [19]:
# абляция: убираем по одной группе признаков из финального набора (на HGB — для скорости)
ablation = {}
for group, cols in FEATURE_GROUPS.items():
    kept = [c for c in X_full.columns if c not in cols]
    ablation[f'без {group} ({len(cols)})'], _, _ = run_experiment(
        hgb(), X_full[kept], {s: Xb[kept] for s, Xb in X_full_blocks.items()})
(pd.DataFrame(ablation).T - pd.Series(results[FINAL_HGB])).round(3).sort_values('P@R сглаж.').add_prefix('Δ ')

,Δ P@R0.7,Δ P@R сглаж.,Δ P@R0.7 по дням,Δ PR-AUC,Δ неделя с 11.04,Δ неделя с 13.04
без pointer (9),-0.093,-0.089,-0.092,-0.039,-0.153,-0.117
без timing (21),-0.073,-0.080,-0.071,-0.042,-0.081,-0.067
без reputation (5),-0.052,-0.037,-0.066,-0.014,-0.088,-0.104
без content (28),-0.016,-0.029,-0.031,-0.017,-0.041,-0.014
без history (4),0.006,-0.004,0.007,0.002,-0.014,-0.014
без cookie (2),-0.009,-0.002,-0.003,0.001,-0.002,0.008
без client (5),0.004,0.002,0.013,-0.003,-0.014,-0.005


In [20]:
# permutation importance финальной модели (один seed) на 17–19 апреля: падение PR-AUC при перемешивании признака
tr, va = (day < '2026-04-17').to_numpy(), (day >= '2026-04-17').to_numpy()
model_late = catboost_model(**CATBOOST_PARAMS).fit(as_catboost_input(X_full[tr]), y[tr])
perm = permutation_importance(model_late, as_catboost_input(X_full[va]), y[va], scoring='average_precision',
                              n_repeats=5, random_state=SEED)
importance = pd.DataFrame({'importance': perm.importances_mean, 'std': perm.importances_std}, index=X_full.columns)
importance['group'] = importance.index.map({c: g for g, cols in FEATURE_GROUPS.items() for c in cols})
importance.sort_values('importance', ascending=False).head(20).round(4)

,importance,std,group
pointer_edge_share,0.0606,0.0057,pointer
item_rep_evidence,0.0359,0.0146,reputation
events_per_active_min,0.0188,0.0057,timing
dt_min,0.0160,0.0021,timing
dt_median,0.0116,0.0013,timing
pointer_x_std,0.0115,0.0056,pointer
item_repeat_share,0.0109,0.0057,content
dt_p10,0.0087,0.0026,timing
category_entropy,0.0069,0.0051,content
share_contact_phone_show,0.0050,0.0022,content


## 8. Сдвиг между train и test

In [21]:
# 1) adversarial validation на поведенческих признаках: ROC-AUC ≈ 0.5 означает, что сдвига поведения нет;
# 2) средние признаков history/reputation по дням трейна и теста: масштаб в тестовую неделю не должен уезжать
is_test = np.r_[np.zeros(len(X)), np.ones(len(X_test))]
adv_score = cross_val_predict(make_hgb(max_iter=200), pd.concat([X[BEHAVIOR], X_test[BEHAVIOR]]), is_test,
                              cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method='predict_proba')[:, 1]
print(f'adversarial ROC-AUC (train против test), поведенческие признаки: {roc_auc_score(is_test, adv_score):.3f}')

cross_day = pd.concat([X_full, X_test_full])[FEATURE_GROUPS['history'] + FEATURE_GROUPS['reputation']]
window_day = meta.set_index('cookie_id').window_start_ts.reindex(cross_day.index).dt.strftime('%d.%m')
cross_day.groupby(window_day.values).mean().loc['13.04':].T.round(3)

adversarial ROC-AUC (train против test), поведенческие признаки: 0.508


,13.04,14.04,15.04,16.04,17.04,18.04,19.04,20.04,21.04,22.04,23.04,24.04,25.04,26.04
hist_pop_mean,0.179,0.187,0.183,0.185,0.184,0.185,0.178,0.184,0.183,0.179,0.171,0.185,0.194,0.187
hist_pop_median,0.156,0.162,0.158,0.158,0.158,0.159,0.148,0.155,0.154,0.143,0.129,0.151,0.158,0.152
hist_pop_max,0.451,0.477,0.447,0.464,0.457,0.483,0.468,0.479,0.468,0.480,0.480,0.480,0.507,0.511
hist_seen_share,0.612,0.633,0.619,0.626,0.612,0.600,0.570,0.593,0.569,0.539,0.516,0.532,0.553,0.554
item_rep_mean,0.113,0.119,0.113,0.115,0.116,0.121,0.110,0.120,0.113,0.113,0.111,0.116,0.122,0.121
item_rep_max,0.263,0.274,0.264,0.272,0.270,0.294,0.276,0.297,0.268,0.281,0.286,0.281,0.292,0.294
item_bot_seen_share,0.110,0.136,0.130,0.140,0.149,0.168,0.145,0.182,0.161,0.160,0.153,0.171,0.187,0.186
item_bot_seen_count,1.327,1.521,1.383,1.452,1.683,1.920,1.580,2.136,1.721,1.917,1.670,1.812,1.988,2.210
item_rep_evidence,-0.713,-0.890,-1.171,-1.304,-1.227,-1.417,-2.109,-1.386,-1.914,-1.511,-2.084,-1.602,-1.413,-1.165


## 9. Отклонённые идеи

In [22]:
# каждая группа добавляется к финальному набору и сравнивается с бустингом в обеих схемах валидации:
# overlap — общие объявления с другими куками того же дня (было в v2); rhythm — форма интервалов внутри сессий
# и время на странице; device — модель устройства и версия ОС; twin — «куки-двойники» с общим набором объявлений;
# репутация других сущностей (UA, город, запрос со страницей, устройство); bigram — переходы между событиями;
# burst — «взрывность» активности; pagination — серии листания выдачи; neighbors — репутация кук-соседей того же дня
def overlap_features(ev, meta):
    views = values_by_day(ev, meta)
    views['others'] = views.groupby(['day', 'item_id']).cookie_id.transform('size') - 1
    views['others_per_1000'] = views.others / views.day.map(meta.groupby('window_start_ts').size()) * 1000
    g = views.groupby('cookie_id')
    return pd.DataFrame({'item_shared_share': g.others.agg(lambda s: (s > 0).mean()),
                         'item_others_mean': g.others_per_1000.mean()})


def rhythm_features(ev):
    dt = ev.groupby('cookie_id').event_ts.diff().dt.total_seconds()
    inside = dt.where(dt <= SESSION_GAP)                 # паузы между сессиями не смешиваем с ритмом
    log_dt = np.log1p(inside).groupby(ev.cookie_id)
    n_inside = inside.notna().groupby(ev.cookie_id).sum().replace(0, np.nan)
    dwell = (-ev.groupby('cookie_id').event_ts.diff(-1).dt.total_seconds()).where(lambda s: s <= SESSION_GAP)
    return pd.DataFrame({
        'in_log_dt_mean': log_dt.mean(), 'in_log_dt_std': log_dt.std(),
        'in_dt_max': inside.groupby(ev.cookie_id).max(),
        'in_dt_share_3_30': inside.between(3, 30, inclusive='left').groupby(ev.cookie_id).sum() / n_inside,
        'in_dt_share_30_300': inside.between(30, 300, inclusive='left').groupby(ev.cookie_id).sum() / n_inside,
        'dwell_item_median': dwell[ev.event_name == 'item_view'].groupby(ev.cookie_id).median(),
        'dwell_search_median': dwell[ev.event_name == 'search_results_view'].groupby(ev.cookie_id).median(),
    })


def device_name(ua: pd.Series) -> pd.Series:
    device = ua.str.extract(r'Android \d+; ([^;)]+)\)').iloc[:, 0]
    return device.fillna(pd.Series(np.where(ua.str.contains('iPhone'), 'iphone', 'none'), index=ua.index))


def device_features(ev):
    ua = ev.groupby('cookie_id').user_agent.first()
    os_version = ua.str.extract(r'Android (\d+)|iOS (\d+)|iPhone OS (\d+)').astype(float).bfill(axis=1).iloc[:, 0]
    return pd.DataFrame({'os_version': os_version, 'device': device_name(ua).astype('category')})


def twin_features(ev, meta):
    views = values_by_day(ev, meta)
    size = views.groupby('cookie_id').size()
    pairs = views.merge(views, on=['day', 'item_id']).query('cookie_id_x != cookie_id_y')
    common = pairs.groupby(['cookie_id_x', 'cookie_id_y']).size().rename('common').reset_index()
    common['jaccard'] = common.common / (common.cookie_id_x.map(size) + common.cookie_id_y.map(size) - common.common)
    g = common.groupby('cookie_id_x')
    return pd.DataFrame({'twin_common_max': g.common.max(), 'twin_jaccard_max': g.jaccard.max(),
                         'n_cookies_sharing_items': g.size()}).reindex(size.index).fillna(0)


def bigram_features(ev, top=15):
    next_event = ev.groupby('cookie_id').event_name.shift(-1)
    pair = (ev.event_name + '>' + next_event).where(next_event.notna())
    n_transitions = pair.notna().groupby(ev.cookie_id).sum().replace(0, np.nan)
    return pd.DataFrame({f'bi_{p}': pair.eq(p).groupby(ev.cookie_id).sum() / n_transitions
                         for p in pair.value_counts().index[:top]})


def burst_features(ev):
    ts = ev.event_ts.astype('int64') // 10**9
    per_minute = ev.assign(minute=ts // 60).groupby(['cookie_id', 'minute']).size().groupby('cookie_id')

    def max_in_60s(t):
        t = np.sort(t.to_numpy())
        return int((np.searchsorted(t, t + 60) - np.arange(len(t))).max())
    return pd.DataFrame({'max_events_60s': ts.groupby(ev.cookie_id).apply(max_in_60s),
                         'active_minutes': per_minute.size(),
                         'busiest_minute_share': per_minute.max() / ev.groupby('cookie_id').size()})


def pagination_features(ev):
    s = ev[ev.event_name == 'search_results_view'][['cookie_id', 'search_query', 'search_page']].copy()
    same_query = s.search_query.eq(s.groupby('cookie_id').search_query.shift())
    s['next_page'] = s.groupby('cookie_id').search_page.diff().eq(1) & same_query
    run_id = (~s.next_page).groupby(s.cookie_id).cumsum()
    max_page = s.groupby(['cookie_id', 'search_query']).search_page.max()
    return pd.DataFrame({'longest_page_run': s.groupby(['cookie_id', run_id]).next_page.sum().groupby('cookie_id').max(),
                         'queries_to_page3': (max_page >= 3).groupby('cookie_id').sum()})


def neighbor_features(ev, meta, reputation):
    views = values_by_day(ev, meta)
    pairs = (views.merge(views, on=['day', 'item_id']).query('cookie_id_x != cookie_id_y')
             [['cookie_id_x', 'cookie_id_y']].drop_duplicates())
    nb = pairs.join(reputation, on='cookie_id_y').join(X_all[FEATURE_GROUPS['history']], on='cookie_id_y')
    g = nb.groupby('cookie_id_x')
    return pd.DataFrame({'nb_count': g.size(), 'nb_bot_seen_mean': g.item_bot_seen_share.mean(),
                         'nb_rep_max': g.item_rep_max.max(), 'nb_hist_pop_mean': g.hist_pop_mean.mean()})


def with_extra(extra_all, extra_blocks=None):
    """Финальный набор + дополнительные признаки (для признаков на метках — свои версии для блоков)."""
    extra_blocks = extra_blocks or {s: extra_all for s in BLOCK_STARTS}
    return (pd.concat([X_full, extra_all.reindex(X.index)], axis=1),
            {s: pd.concat([X_full_blocks[s], extra_blocks[s].reindex(X.index)], axis=1) for s in BLOCK_STARTS})


rejected = {}
overlap = overlap_features(events, meta)
for name, extra in [('overlap', overlap), ('rhythm', rhythm_features(events)),
                    ('device', device_features(events)), ('twin', twin_features(events, meta))]:
    rejected[f'+ {name}'], _, _ = run_experiment(hgb(), *with_extra(extra))

events_ext = events.assign(device=device_name(events.user_agent).where(events.platform != 'web'),
                           query_page=events.search_query + ' | ' + events.search_page.astype('Int64').astype(str))
for column, prefix in [('user_agent', 'ua'), ('item_location', 'location'), ('query_page', 'query_page'), ('device', 'device')]:
    rep_all = reputation_features(events_ext, meta, train_labels, column, prefix)
    rep_blocks = {s: reputation_features(events_ext, meta, train_labels[label_day < s], column, prefix) for s in BLOCK_STARTS}
    rejected[f'+ репутация {prefix}'], _, _ = run_experiment(hgb(), *with_extra(rep_all, rep_blocks))

for name, extra in [('bigram', bigram_features(events)), ('burst', burst_features(events)),
                    ('pagination', pagination_features(events))]:
    rejected[f'+ {name}'], _, _ = run_experiment(hgb(), *with_extra(extra))
rejected['+ neighbors'], _, _ = run_experiment(hgb(), *with_extra(
    neighbor_features(events, meta, REPUTATION['все метки']),
    {s: neighbor_features(events, meta, REPUTATION[s]) for s in BLOCK_STARTS}))
(pd.DataFrame(rejected).T - pd.Series(results[FINAL_HGB])).round(3).add_prefix('Δ ')

,Δ P@R0.7,Δ P@R сглаж.,Δ P@R0.7 по дням,Δ PR-AUC,Δ неделя с 11.04,Δ неделя с 13.04
+ overlap,0.006,0.003,0.001,0.002,-0.019,-0.009
+ rhythm,-0.019,-0.006,-0.023,0.005,-0.009,-0.004
+ device,0.001,0.006,-0.010,0.005,0.005,0.002
+ twin,-0.009,-0.003,-0.023,0.001,0.000,-0.011
+ репутация ua,0.008,0.015,0.000,0.006,0.003,-0.010
+ репутация location,-0.002,-0.003,-0.006,0.001,0.000,-0.009
+ репутация query_page,-0.008,0.002,-0.010,0.003,0.002,-0.002
+ репутация device,-0.008,0.006,-0.017,0.004,-0.011,0.005
+ bigram,0.001,0.001,-0.012,0.006,0.000,0.002
+ burst,0.004,0.003,-0.012,0.003,0.000,0.011


In [23]:
# постобработка: ранги внутри дня против глобального порядка (на недельных блоках финальной модели)
within_day = {}
for start in BLOCK_STARTS:
    _, va = block_masks(start)
    score = block_scores[FINAL][start]
    within_day[f'неделя с {start:%d.%m}'] = {
        'глобальный порядок': precision_at_recall(y[va], score),
        'ранги внутри дня': precision_at_recall(y[va], pd.Series(score).groupby(day[va].to_numpy()).rank(pct=True).to_numpy()),
        'доля ботов по дням, мин–макс': f'{pd.Series(y[va]).groupby(day[va].to_numpy()).mean().min():.3f}–{pd.Series(y[va]).groupby(day[va].to_numpy()).mean().max():.3f}'}
pd.DataFrame(within_day).T

,глобальный порядок,ранги внутри дня,"доля ботов по дням, мин–макс"
неделя с 11.04,0.891967,0.875339,0.063–0.093
неделя с 13.04,0.894081,0.864458,0.063–0.093


In [24]:
# прошлые версии решения: v2 — ансамбль на поведении + overlap; v3 — ансамбль на финальных признаках
# без суммы свидетельств в reputation
V3_COLUMNS = [c for c in X_full.columns if c not in ('item_bot_seen_count', 'item_rep_evidence')]
evaluate('v2: ансамбль, поведение + overlap', ensemble, pd.concat([X[BEHAVIOR], overlap.reindex(X.index)], axis=1))
evaluate('v3: ансамбль, без суммы свидетельств', ensemble, X_full[V3_COLUMNS],
         {s: Xb[V3_COLUMNS] for s, Xb in X_full_blocks.items()})
pd.concat({'финальная против v2': bootstrap_diff(FINAL, 'v2: ансамбль, поведение + overlap'),
           'финальная против v3': bootstrap_diff(FINAL, 'v3: ансамбль, без суммы свидетельств')})

разница     от     до  P(разница > 0)
финальная против v2 OOF 8 дней        0.125  0.072  0.187           1.000
                    неделя с 11.04    0.120  0.062  0.189           1.000
                    неделя с 13.04    0.113  0.066  0.171           1.000
финальная против v3 OOF 8 дней        0.024 -0.004  0.059           0.954
                    неделя с 11.04    0.034 -0.007  0.072           0.950
                    неделя с 13.04    0.006 -0.021  0.033           0.693

## 10. Разбор ошибок

In [25]:
# OOF-предсказания финальной модели при пороге, на котором достигается метрика
def threshold_at_recall(y_true, score, recall=0.7):
    """Порог, на котором достигается официальная метрика (группы равных score не делятся)."""
    s = np.sort(score)[::-1]
    thresholds = s[np.r_[s[1:] != s[:-1], True]]
    prec, rec = pr_curve(y_true, score)
    ok = rec >= recall
    return thresholds[ok][np.argmax(prec[ok])]


final_oof = oof[FINAL].dropna()
err = X_full.loc[final_oof.index].assign(target=y[oof[FINAL].notna().to_numpy()], score=final_oof)
t = threshold_at_recall(err.target, err.score)
err['flagged'] = err.score >= t
print(f'отмечено {err.flagged.sum()} кук, из них ботов {(err.flagged & err.target.eq(1)).sum()} '
      f'из {err.target.sum()}, precision {precision_at_recall(err.target, err.score):.3f}')


def error_breakdown(by):
    return err.groupby(by, observed=True).apply(lambda d: pd.Series({
        'кук': len(d), 'ботов': d.target.sum(),
        'recall': d.flagged[d.target.eq(1)].mean(),
        'людей отмечено': (d.flagged & d.target.eq(0)).sum(),
    })).round(3)


# на android различаем приложение Авито и мобильный браузер — это разные сегменты
err['segment'] = np.where(err.platform == 'android', 'android: ' + err.ua_client.astype(str), err.platform.astype(str))
error_breakdown('segment')

отмечено 368 кук, из них ботов 336 из 475, precision 0.913


,кук,ботов,recall,людей отмечено
segment,,,,
android: app,935.0,100.0,0.660,1.0
android: mobile_browser,1619.0,50.0,0.320,8.0
ios,286.0,11.0,0.364,0.0
web,3138.0,314.0,0.796,23.0


In [26]:
# чем пропущенные боты отличаются от найденных и от людей
bots = err[err.target.eq(1)]
compare = ['n_events', 'dt_median', 'engagement_share', 'location_nunique', 'pointer_share', 'pointer_x_std',
           'cookie_age_days', 'hist_pop_mean', 'item_rep_mean', 'item_bot_seen_share']
pd.concat({'найденные боты': bots[bots.flagged][compare].median(),
           'пропущенные боты': bots[~bots.flagged][compare].median(),
           'люди': err[err.target.eq(0)][compare].median()}, axis=1).round(3)

,найденные боты,пропущенные боты,люди
n_events,25.500,14.000,12.000
dt_median,20.000,37.750,60.500
engagement_share,0.143,0.222,0.263
location_nunique,10.000,6.000,5.000
pointer_share,0.000,0.000,0.938
pointer_x_std,187.597,341.853,552.003
cookie_age_days,2.622,45.944,62.791
hist_pop_mean,0.369,0.171,0.167
item_rep_mean,0.341,0.092,0.088
item_bot_seen_share,0.778,0.083,0.000


In [27]:
# боты train, разбитые на 4 кластера по поведению (без платформы и history/reputation): похоже на сервисы из условия
CLUSTER_FEATURES = ['dt_median', 'dt_p10', 'dt_p90', 'events_per_active_min', 'n_events', 'n_sessions', 'night_share',
                    'share_search_results_view', 'share_item_view', 'share_photo_swipe', 'share_seller_page_view',
                    'share_contact_phone_show', 'share_favorite_add', 'search_page_mean', 'location_nunique',
                    'category_nunique', 'item_repeat_share', 'cookie_age_days']
bot_rows = X_full[y == 1][CLUSTER_FEATURES]
bot_z = StandardScaler().fit_transform(np.log1p(bot_rows.fillna(bot_rows.median()).clip(lower=0)))
bot_cluster = pd.Series(KMeans(4, n_init=20, random_state=SEED).fit_predict(bot_z), index=bot_rows.index)
# имена кластеров — интерпретация по профилю (по какому признаку кластер выделяется сильнее остальных)
medians = bot_rows.groupby(bot_cluster).median()
names = {medians.share_contact_phone_show.idxmax(): 'выгрузка контактов', medians.dt_median.idxmax(): 'мониторинг цен',
         medians.search_page_mean.idxmax(): 'сводки по категориям', medians.cookie_age_days.idxmax(): 'имитация человека'}
if len(names) == 4:
    bot_cluster = bot_cluster.map(names)

bot_segment = np.where(X_full.platform[y == 1] == 'android', 'android: ' + X_full.ua_client[y == 1].astype(str),
                       X_full.platform[y == 1].astype(str))
profile = bot_rows.groupby(bot_cluster)[['dt_median', 'n_events', 'search_page_mean', 'location_nunique',
                                         'share_contact_phone_show', 'share_photo_swipe', 'cookie_age_days']].median().T
profile.loc['ботов'] = bot_cluster.value_counts().sort_index()
profile.loc['recall (OOF, 12–19 апр)'] = err.flagged[err.target.eq(1)].groupby(bot_cluster).mean()
display(profile.round(2))
pd.crosstab(bot_cluster.rename('кластер'), pd.Series(bot_segment, index=bot_rows.index, name='сегмент'))

,выгрузка контактов,имитация человека,мониторинг цен,сводки по категориям
dt_median,21.50,35.00,384.00,12.00
n_events,13.00,19.00,7.00,45.00
search_page_mean,2.00,2.87,2.67,5.25
location_nunique,6.00,8.00,4.00,14.00
share_contact_phone_show,0.10,0.00,0.00,0.00
share_photo_swipe,0.04,0.10,0.00,0.06
cookie_age_days,2.40,55.85,2.40,2.16
ботов,183.00,292.00,141.00,283.00
"recall (OOF, 12–19 апр)",0.65,0.54,0.68,0.94


сегмент,android: app,android: mobile_browser,ios,web
кластер,,,,
выгрузка контактов,102,22,5,54
имитация человека,49,50,15,178
мониторинг цен,12,20,1,108
сводки по категориям,17,13,5,248


In [28]:
# сколько кук пришлось бы отметить, чтобы отдельным порогом найти 70% ботов мобильного браузера
mobile_browser = err[err.segment == 'android: mobile_browser']
bot_scores = np.sort(mobile_browser.score[mobile_browser.target.eq(1)].to_numpy())[::-1]
t_mb = bot_scores[int(np.ceil(0.7 * len(bot_scores))) - 1]
flag_mb = mobile_browser.score >= t_mb
print(f'чтобы найти 70% ботов мобильного браузера, пришлось бы отметить {flag_mb.sum()} кук этого сегмента, '
      f'из них людей {(flag_mb & mobile_browser.target.eq(0)).sum()}')

чтобы найти 70% ботов мобильного браузера, пришлось бы отметить 476 кук этого сегмента, из них людей 441


## 11. Сводная таблица

In [29]:
show_results(['константа', 'baseline из quickstart (RF на 2 признаках)', 'эвристика: -dt_median', V1,
              'v2: ансамбль, поведение + overlap', 'v3: ансамбль, без суммы свидетельств',
              'бустинг: поведение', 'бустинг: поведение + history', FINAL_HGB, 'ансамбль', FINAL])

,P@R0.7,P@R сглаж.,P@R0.7 по дням,PR-AUC,неделя с 11.04,неделя с 13.04
константа,0.079,0.079,0.080,0.079,0.082,0.079
baseline из quickstart (RF на 2 признаках),0.093,0.095,0.097,0.206,0.096,0.093
эвристика: -dt_median,0.184,0.177,0.173,0.299,0.154,0.179
"v1: бустинг с подобранными параметрами, только поведение",0.760,0.733,0.755,0.779,0.749,0.728
"v2: ансамбль, поведение + overlap",0.778,0.769,0.796,0.799,0.772,0.786
"v3: ансамбль, без суммы свидетельств",0.888,0.845,0.909,0.828,0.847,0.889
бустинг: поведение,0.751,0.738,0.754,0.786,0.716,0.737
бустинг: поведение + history,0.849,0.825,0.846,0.814,0.771,0.788
бустинг: поведение + history + reputation,0.902,0.862,0.912,0.828,0.859,0.892
ансамбль,0.889,0.854,0.922,0.831,0.885,0.897
